In [ ]:
"""
Week 3 - Day 2
Full PPO Training
==================
Training PPO agent and comparing
with DQN and all baselines.

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from environment.pricing_env import DynamicPricingEnv
from agents.ppo.ppo_agent import PPOAgent
from agents.dqn.dqn_agent import DQNAgent
from agents.q_learning_agent import (
    QLearningAgent, QL_CONFIG
)
from training.ppo_trainer import (
    plot_ppo_training,
    compare_all_agents
)
from analysis.ppo_vs_dqn import (
    compare_stability,
    compare_trajectories,
    statistical_ppo_vs_dqn
)
from config import PPO, DQN

plt.style.use('seaborn-v0_8')
print("✅ PPO Training modules loaded!")

In [ ]:
env = DynamicPricingEnv()

# Train PPO
print("Training PPO (2000 episodes)...")
ppo_agent = PPOAgent(env, PPO)
ppo_agent.train(n_episodes=2000, verbose=True)
ppo_eval  = ppo_agent.evaluate(n_episodes=100)
print(f"✅ PPO: ${ppo_eval['mean_revenue']:.0f}")

# Train DQN
print("\nTraining DQN (2000 episodes)...")
dqn_agent = DQNAgent(env, DQN)
dqn_agent.train(n_episodes=2000, verbose=False)
dqn_eval  = dqn_agent.evaluate(n_episodes=100)
print(f"✅ DQN: ${dqn_eval['mean_revenue']:.0f}")

# Train Q-Learning
print("\nTraining Q-Learning (3000 episodes)...")
ql_agent = QLearningAgent(env, QL_CONFIG)
ql_agent.train(n_episodes=3000, verbose=False)
ql_eval  = ql_agent.evaluate(n_episodes=100)
print(f"✅ Q-Learning: ${ql_eval['mean_revenue']:.0f}")

In [ ]:
plot_ppo_training(
    ppo_agent,
    save_path='../results/ppo_training.png'
)

In [ ]:
all_results, ranked = compare_all_agents(
    env, ppo_agent, dqn_agent, ql_agent,
    n_eval=100,
    save_path='../results/all_agents_comparison.png'
)

In [ ]:
ppo_cv, dqn_cv = compare_stability(
    ppo_agent.episode_rewards,
    dqn_agent.episode_rewards,
    save_path='../results/stability_comparison.png'
)

In [ ]:
compare_trajectories(
    ppo_agent, dqn_agent, env,
    save_path='../results/ppo_dqn_trajectories.png'
)

In [ ]:
stats_results = statistical_ppo_vs_dqn(
    ppo_agent, dqn_agent, env,
    n_episodes=200
)

In [ ]:
print("=== FINAL COMPARISON TABLE ===\n")
print(f"{'Agent':<15} {'Revenue':>10} {'Rank'}")
print("─" * 35)
medals = ['🥇', '🥈', '🥉',
          '4️⃣', '5️⃣', '6️⃣', '7️⃣']
for i, (name, rev) in enumerate(ranked):
    print(f"  {medals[i]} {name:<15}: ${rev:.0f}")

best_bl = max(
    v for k, v in all_results.items()
    if k not in ['PPO 🏆', 'DQN', 'Q-Learning']
)
ppo_rev = all_results['PPO 🏆']
imp_vs_bl  = (ppo_rev - best_bl) / best_bl * 100
imp_vs_dqn = (
    ppo_rev - all_results['DQN']
) / all_results['DQN'] * 100

print(f"\n  PPO vs Best Baseline: {imp_vs_bl:+.1f}%")
print(f"  PPO vs DQN          : {imp_vs_dqn:+.1f}%")

In [ ]:
ppo_rank = next(
    i+1 for i, (n, _) in enumerate(ranked)
    if 'PPO' in n
)

print("╔══════════════════════════════════════════╗")
print("║    WEEK 3 DAY 2 — PPO TRAINING DONE!    ║")
print("╠══════════════════════════════════════════╣")
print("║  PPO RESULTS:                            ║")
print(f"║  Mean Revenue : "
      f"${ppo_eval['mean_revenue']:.0f}"
      f"{'':<22} ║")
print(f"║  vs DQN       : {imp_vs_dqn:+.1f}%"
      f"{'':<24} ║")
print(f"║  vs Baseline  : {imp_vs_bl:+.1f}%"
      f"{'':<24} ║")
print(f"║  Final Rank   : #{ppo_rank}"
      f"{'':<24} ║")
print("╠══════════════════════════════════════════╣")
print("║  STABILITY:                              ║")
print(f"║  PPO CV : {ppo_cv:.4f}"
      f"{'':<27} ║")
print(f"║  DQN CV : {dqn_cv:.4f}"
      f"{'':<27} ║")
if ppo_cv < dqn_cv:
    print("║  ✅ PPO is MORE stable than DQN!        ║")
else:
    print("║  ✅ Both agents performing well!         ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Hyperparameter Tuning 🔧     ║")
print("╚══════════════════════════════════════════╝")